# Teste da U-Net - Monitor do Fogo - Brasil - Pantanal

**Dados:** `brazil / MONITOR_01` | **Sensor:** Sentinel-2 (10 m) | **Mosaico:** minnbr, mensal | **Regiao:** Pantanal

Experimento independente: uma **U-Net** treinada com recortes de imagem (chips), como alternativa
a rede MLP por pixel do catalogo (M4/M5). Ela le os **mesmos mosaicos mensais** (COGs do catalogo M1/M2)
e os **mesmos poligonos mensais de fogo / nao-fogo** (CSVs do catalogo M3) no Cloud Storage,
e nao importa nada das versoes do catalogo.

Codigo: `network/catalog/experiments/unet_pantanal/` (branch `classification-unet`).
Guia: `README.md` nessa pasta.

**Use um ambiente novo com GPU** (Ambiente de execucao > Alterar tipo > GPU). Nao rode o treinador M4
do catalogo no mesmo ambiente: ele muda o TensorFlow para o modo v1.

#### Passos

| Passo | O que faz | Saida (GCS `.../MONITOR_01/UNET_01/`) |
|---|---|---|
| 0 | Preparacao, login e configuracao | - |
| 1 | Inventario: confere mosaico e amostras de cada mes | resumo na tela |
| 2 | Blocos espaciais + chips + rotulos (1 fogo / 0 nao-fogo / 255 desconhecido) | `chips/`, `splits/`, `reports/chips_*` |
| 3 | Estatisticas de normalizacao (so chips de treino) | `stats/norm_stats.json` |
| 4 | Treina uma U-Net com os chips de todos os meses em `PERIODS` | `models/` |
| 5 | Metricas de teste (limiar escolhido na validacao) | `reports/metrics_*.json` |
| 6 | Classifica a regiao, um mapa por mes em `PREDICT_PERIODS` | `predict/{regiao}_{periodo}_prob.tif` |
| 7 | Compara com o MLP (A bandas, B bandas + indices, C U-Net) | `reports/comparison_*.json` |

## [0] Preparacao

In [ ]:
# 0.1 -- Preparacao do Colab (clona a branch, instala, ajusta caminhos)
import sys, os, subprocess, shutil

BRANCH = "classification-unet"
REPO_DIR = "/content/brazil-fire"
os.chdir("/content")
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                "https://github.com/mapbiomas/brazil-fire.git", REPO_DIR], check=True)

EXP_DIR = os.path.join(REPO_DIR, "network/catalog/experiments/unet_pantanal")
subprocess.run(["pip", "install", "-q", "-r", os.path.join(EXP_DIR, "requirements.txt")], check=True)
sys.path.insert(0, os.path.join(EXP_DIR, "src"))
# descarta modulos unet_* ja importados nesta sessao (senao o Python usa a versao antiga)
for _m in [m for m in sys.modules if m.startswith("unet_")]:
    del sys.modules[_m]

In [ ]:
# 0.2 -- Configuracao da U-Net + login (todos os caminhos saem daqui)
import unet_env
from unet_config import make_config, paths

UCFG = make_config(
    # projeto (onde estao os dados do catalogo)
    COUNTRY="brazil",
    BUCKET="mapbiomas-fire",
    GCS_PREFIX="sudamerica/brazil/CATALOG_01",
    CAMPAIGN="MONITOR_01",
    GEE_PROJECT="mapbiomas-brazil",
    # asset de regioes: region_name = bioma + "_r" + id (calculado, como no script do GEE)
    REGIONS_ASSET="users/geomapeamentoipam/AUXILIAR/regioes_biomas_col2",
    REGION_PROPERTY="region_name", REGION_NAME_FIELDS=["bioma", "id"], REGION_NAME_JOIN="_r",
    # mosaicos: LIBRARY_IMAGES/SENTINEL2/MONTHLY/MINNBR/{periodo}/COG/
    SENSOR="sentinel2",
    MOSAIC="minnbr",
    PERIODS=["2026_10"],             # meses com mosaico E amostras (treino); formato 'AAAA_MM'
    PREDICT_PERIODS=["2026_10"],     # meses a classificar (precisam so do mosaico)
    # experimento
    EXPERIMENT="UNET_01",            # pasta no GCS, dentro da campanha (saidas)
    MODEL_NAME="unet_monitor_pantanal_v1",
    REGION="Pantanal_r1",            # region_name a classificar (se errar, o erro lista as opcoes)
    SAMPLE_REGION="Pantanal 60",     # valor da coluna `region` nos CSVs de amostras
    # Exportadas para 2026_10: blue, green, red, swir2, dayOfYear (ainda sem nir/swir1).
    BANDS=["blue", "green", "red", "swir2"],   # o que ja foi exportado
    # BANDS=["green", "swir2"],   # primeiro teste
    # BANDS=["blue", "red", "nir", "swir1", "swir2"],   # conjunto completo, quando exportado
    INDICES=[],                      # sem nir/swir1 ainda, entao sem indices
    # INDICES=["nbr", "nbr2", "mirbi", "ndvi", "bai", "evi"],   # precisa do conjunto completo
    VALID_BAND="dayOfYear",          # exportada: mascara de pixel valido + banda 2 da saida (None = sem ela)
    SAMPLE_GLOB="*{period}*.csv",    # CSVs M3 em MONITOR_01/LIBRARY_SAMPLES; ou SAMPLE_FILES=[...]
    CHIP=256, POINTS_PER_POLYGON=4, BLOCK_KM=25, SPLIT=(0.70, 0.15, 0.15),
    BATCH=8, EPOCHS=60, LR=1e-4, SEED=42,
    OVERLAP=16, INFER_TILE=512,
)
P = paths(UCFG)
unet_env.connect(UCFG)               # pop-up do Colab; GEE com a sua conta, GCS com a conta de servico
print(f"Saidas:   gs://{P['bucket']}/{P['root']}/")
print(f"Amostras: gs://{P['bucket']}/{P['samples']}/")

## [1] Inventario (so leitura)
Nao treina nada. Confere se cada banda existe e esta na mesma grade, lista os arquivos de amostras e mostra um resumo (bandas, poligonos por classe, o que observar).

In [ ]:
import json
from unet_io import MosaicReader, load_samples, list_sample_files, print_inventory

MOSTRAR_JSON = False   # True = tambem imprime o inventario bruto (JSON)

for period in UCFG["PERIODS"]:
    reader = MosaicReader(UCFG, period)
    inv = reader.inventory()
    sample_files = list_sample_files(UCFG, period)
    samples, report = load_samples(UCFG, period, reader.crs)
    print_inventory(UCFG, inv, sample_files, report)
    if MOSTRAR_JSON:
        print(json.dumps(inv, indent=2, default=str))
    reader.close()

## [2] Chips e rotulos
Os blocos de `BLOCK_KM` sao divididos em treino/validacao/teste **antes** de fazer os chips.
Um chip que cruza blocos de divisoes diferentes e descartado. Rotulo: 1 fogo, 0 nao-fogo,
255 fora dos poligonos ou nodata (a perda ignora). Onde os poligonos se sobrepoem, vale o fogo.

In [ ]:
from IPython.display import Image, display
from unet_chips import build_chips
from unet_io import local_dir
from unet_resumo import resumo_chips

# True = apaga e refaz os chips do periodo. Necessario depois de mudar bandas, amostras,
# regiao ou BLOCK_KM (os chips salvos no GCS sao de uma execucao anterior).
REFAZER_CHIPS = True

for period in UCFG["PERIODS"]:
    summary = build_chips(UCFG, period, overwrite=REFAZER_CHIPS)
    resumo_chips(UCFG, summary)
    if not summary.get("skipped"):
        png = os.path.join(local_dir("reports"), f"chips_preview_{period}.png")
        if os.path.exists(png):
            display(Image(png))

Confira a figura: vermelho = fogo, azul = nao-fogo. Atencao a chips sobre lagoas e areas alagadas (agua escura parece cicatriz de fogo) e sobre nuvens ou sombras de nuvem que restaram no mosaico mensal.

## [3] Estatisticas de normalizacao (so chips de treino)

In [ ]:
from unet_chips import compute_stats
from unet_resumo import resumo_estatisticas

stats = compute_stats(UCFG)
resumo_estatisticas(stats)

## [4] Treino da U-Net

In [ ]:
from unet_model import train
from unet_resumo import resumo_treino

model, meta = train(UCFG)
resumo_treino(UCFG, meta)

## [5] Metricas de teste
O limiar com melhor F1 e escolhido na **validacao**; o teste e reportado nesse limiar e em 0.5.

In [ ]:
from unet_model import load_model
from unet_report import evaluate_unet
from unet_resumo import resumo_teste

model = load_model(UCFG)
unet_results = evaluate_unet(UCFG, model)
resumo_teste(unet_results)

## [6] Classificar a regiao inteira
Janela deslizante: cada janela le `INFER_TILE + 2*OVERLAP` pixels e guarda so o centro,
para nao aparecerem emendas. Saida no formato M5 (probabilidade x 1000 + dayOfYear,
nodata -9999), na mesma grade do mosaico.

In [ ]:
from unet_io import read_json
from unet_predict import predict_mosaic, gee_upload_command
from unet_config import region_slug

stats = read_json(P["stats"])
for period in UCFG["PREDICT_PERIODS"]:
    res = predict_mosaic(UCFG, period, model, stats)
    print(f"Mapa de {period} salvo em {res['gcs']}")
    print(f"   {res['valid_pixels']:,} pixels validos | {res['burned_pixels_prob_gt_0']:,} com probabilidade > 0 "
          f"| {res['minutes']} min")
    print("   Para subir ao GEE (edite o caminho do asset antes):")
    print("   " + gee_upload_command(UCFG, period, f"projects/{UCFG['GEE_PROJECT']}/assets/FIRE/CATALOG_01/"
                                  f"{UCFG['CAMPAIGN']}/{UCFG['EXPERIMENT']}/{region_slug(UCFG)}_{period}_prob"))

## [7] Comparacao com o MLP
Mesmos pixels de teste para todos os modelos. A = MLP com bandas, B = MLP com bandas + indices (pulado quando `INDICES` esta vazio), C = U-Net.

In [ ]:
from IPython.display import Markdown
from unet_report import compare_with_mlp
rows, table = compare_with_mlp(UCFG, unet_results)
display(Markdown(table))